# Build an AI Agent — runnable notebook

This notebook is a Colab/Kaggle/Binder-friendly version of the **Build an AI Agent** project from the
[Python & Data Analysis course](https://github.com/abderrahim-lectures/python-data-analysis-course). It mirrors the
real, runnable example at [`examples/ai-agent/agent.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/ai-agent/agent.py) —
a small tool-calling agent built with LangChain's [`deepagents`](https://github.com/langchain-ai/deepagents).

You'll need a free-tier API key from one of six providers — no credit card required for any of them. See the
[project doc](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/docs/projects/ai-agent/index.md#step-2-get-a-free-ai-api-key)
for where to get a key for each provider.

Run the cells in order.

In [ ]:
!pip install deepagents langchain-openai python-dotenv

## Pick a provider and enter your API key

**You're free to use whichever provider you like.** GitHub Models is the suggested default below since it needs no
separate signup (you already have a GitHub account), but Gemini, Groq, Mistral, Cerebras, and OpenRouter all have
workable free tiers too. `getpass` is used instead of a plain `input()` so the key never gets echoed to the notebook
output or saved into its cell history.

In [ ]:
import os
from getpass import getpass

# One of: "github" (default), "gemini", "groq", "mistral", "cerebras", "openrouter"
PROVIDER = "github"

ENV_VAR_BY_PROVIDER = {
    "github": "GITHUB_TOKEN",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "cerebras": "CEREBRAS_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
}

if PROVIDER not in ENV_VAR_BY_PROVIDER:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'. Choose one of: {', '.join(ENV_VAR_BY_PROVIDER)}")

env_var = ENV_VAR_BY_PROVIDER[PROVIDER]
os.environ[env_var] = getpass(f"Enter your {PROVIDER} API key (stored only in {env_var} for this session): ")

## Build the model

Every provider here is OpenAI-compatible or has its own small LangChain integration package. This mirrors
`_build_*_model()` in `agent.py` — GitHub Models and Cerebras/OpenRouter reuse `ChatOpenAI` with a different
`base_url`; Gemini, Groq, and Mistral use their own packages (installed on demand below if you picked one of
those providers).

In [ ]:
def build_model(provider: str):
    if provider == "github":
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model="gpt-4o-mini",  # confirm this still has a free tier before relying on it
            api_key=os.environ["GITHUB_TOKEN"],
            base_url="https://models.github.ai/inference",
        )
    if provider == "gemini":
        %pip install -q langchain-google-genai
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model="gemini-3.5-flash",  # pinned, versioned model ID -- not a "-latest" alias
            google_api_key=os.environ["GOOGLE_API_KEY"],
        )
    if provider == "groq":
        %pip install -q langchain-groq
        from langchain_groq import ChatGroq

        return ChatGroq(model="llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])
    if provider == "mistral":
        %pip install -q langchain-mistralai
        from langchain_mistralai import ChatMistralAI

        return ChatMistralAI(model="mistral-small-latest", api_key=os.environ["MISTRAL_API_KEY"])
    if provider == "cerebras":
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model="llama-3.3-70b",
            api_key=os.environ["CEREBRAS_API_KEY"],
            base_url="https://api.cerebras.ai/v1",
        )
    if provider == "openrouter":
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model="meta-llama/llama-3.3-70b-instruct:free",
            api_key=os.environ["OPENROUTER_API_KEY"],
            base_url="https://openrouter.ai/api/v1",
        )
    raise ValueError(f"Unknown provider '{provider}'")


model = build_model(PROVIDER)
model

## Define the toy tools

These two functions are deliberately trivial -- the same toy tools from the project doc's `agent.py` walkthrough.
The docstring on each one (the triple-quoted string right after `def`) is what the agent reads to decide which
tool matches which question -- there's no `if`/`elif` routing written by hand.

In [ ]:
def search_course_topics(query: str) -> str:
    """A toy tool: pretends to look up whether a topic was covered in this course."""
    topics = ["variables", "loops", "functions", "csv files", "pandas", "dataframes", "groupby"]
    matches = [t for t in topics if query.lower() in t]
    return f"Matching topics: {matches}" if matches else "No matching topics found."


def count_weeks_remaining(current_week: int) -> str:
    """A second toy tool: how many weeks are left in the 10-week course."""
    remaining = max(0, 10 - current_week)
    return f"{remaining} week(s) remaining out of 10."

## Wire the agent together

`create_deep_agent` combines the model with a list of Python functions the agent can call as **tools** -- a
language model that can decide to call your code, read the result, and use it to answer, instead of only
responding with text.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    tools=[search_course_topics, count_weeks_remaining],
    system_prompt="You help students figure out whether a topic was covered in their course.",
)

## Run it on a real sample question

Just the final answer, not the full internal trace -- see the project doc's "Understanding the full internal
trace" section if you want to see every intermediate tool call.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "Did we cover groupby?"}]})
print(result["messages"][-1].content)